# Sovereign AI: Fine-Tuning TinyAya on Yoruba for Constrained Kubernetes

This notebook trains a localized LoRA adapter on top of **CohereLabs/tiny-aya-earth** (3.35B parameters) using the **masakhane/african-ultrachat** (Yoruba split) dataset.

### Pipeline Stages:
1. **Baseline Evaluation**: Test base `tiny-aya-earth` generation on authentic Yoruba prompts.
2. **QLoRA Fine-Tuning**: Efficient 4-bit fine-tuning using `peft` + `trl.SFTTrainer`.
3. **Post-Training Evaluation**: Re-test with identical Yoruba prompts to measure linguistic and tonal improvements.
4. **Export & Upload**: Push trained adapter to Hugging Face Hub (`husseinalamutu/tiny-aya-earth-yoruba-lora`).
5. **Merge Weights**: Consolidate LoRA adapter into full FP16 weights ready for GGUF conversion.

In [ ]:
# 1. Install dependencies
!pip install -q -U torch transformers peft trl datasets bitsandbytes accelerate huggingface_hub

In [ ]:
# 2. Verify GPU allocation
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. Switch Colab runtime to GPU (T4 or A100) via Runtime -> Change runtime type.")

In [ ]:
# 3. Optional: Authenticate to Hugging Face (Required to push adapters/models)
from huggingface_hub import login
import getpass

# Enter your write-enabled HF token
hf_token = getpass.getpass("Enter your Hugging Face Access Token (optional if not pushing): ")
if hf_token.strip():
    login(token=hf_token.strip())

In [ ]:
# 4. Load Base Model and Run Baseline Evaluation (Before Training)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "CohereLabs/tiny-aya-earth"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base tokenizer and model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

# Test Baseline Prompt (Capture this for the conference talk artifacts!)
test_prompt = "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>Bawo ni o se le se alaye bi ero ayelujara (Internet) se n sise ni ede Yoruba to rorun?<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

print("Generating baseline output...")
with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

baseline_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- BASELINE OUTPUT (Before Fine-Tuning) ---")
print(baseline_response)
print("-------------------------------------------\n")

In [ ]:
# 5. Prepare African-UltraChat Yoruba Dataset
from datasets import load_dataset

dataset_name = "masakhane/african-ultrachat"
print(f"Loading dataset: {dataset_name}...")

try:
    dataset = load_dataset(dataset_name, "yo", split="train")
except Exception:
    dataset = load_dataset(dataset_name, split="train")
    if "language" in dataset.column_names:
        dataset = dataset.filter(lambda x: x["language"].lower() in ["yo", "yoruba"])

# Take a focused sample of 3000-5000 dialogues for Colab Pro execution
max_train_samples = min(4000, len(dataset))
dataset = dataset.shuffle(seed=42).select(range(max_train_samples))
print(f"Training with {len(dataset)} Yoruba conversation records.")

def format_example(record):
    # Standard African-UltraChat conversation format
    text = ""
    if "messages" in record:
        for m in record["messages"]:
            role = m["role"]
            content = m["content"].strip()
            if role == "user":
                text += f"<|START_OF_TURN_TOKEN|><|USER_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
            elif role == "assistant":
                text += f"<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
    return {"text": text}

formatted_ds = dataset.map(format_example)
print("Sample formatted record:", formatted_ds[0]["text"][:200])

In [ ]:
# 6. Configure LoRA and Train with SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./tiny-aya-earth-yoruba-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_ds,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting LoRA training on Yoruba split...")
trainer.train()

# Save LoRA adapter locally
trainer.model.save_pretrained("./tiny-aya-earth-yoruba-lora")
tokenizer.save_pretrained("./tiny-aya-earth-yoruba-lora")
print("Adapter saved to ./tiny-aya-earth-yoruba-lora")

In [ ]:
# 7. Post-Training Evaluation: Test Identical Prompt
peft_model.eval()
print("Generating fine-tuned output with identical prompt...")
with torch.no_grad():
    outputs = peft_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

finetuned_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- FINE-TUNED OUTPUT (After LoRA Training) ---")
print(finetuned_response)
print("-----------------------------------------------\n")

In [ ]:
# 8. Push Adapter to Hugging Face Hub (Optional/Recommended)
hub_adapter_id = "husseinalamutu/tiny-aya-earth-yoruba-lora"
try:
    print(f"Pushing adapter to {hub_adapter_id}...")
    trainer.model.push_to_hub(hub_adapter_id)
    tokenizer.push_to_hub(hub_adapter_id)
    print("Successfully uploaded to Hugging Face!")
except Exception as e:
    print(f"Push to hub skipped or failed: {e}")

In [ ]:
# 9. Merge LoRA Weights into Base Model and Save for GGUF Conversion
from peft import PeftModel
import gc

print("Consolidating weights: loading base model in FP16...")
# Clear VRAM
del base_model, peft_model, trainer
gc.collect()
torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu", # Merge in CPU RAM to prevent GPU OOM
    trust_remote_code=True,
)

print("Loading trained LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_fp16, "./tiny-aya-earth-yoruba-lora")
merged_model = merged_model.merge_and_unload()

merged_output_dir = "./tiny-aya-earth-yoruba-merged"
print(f"Saving merged weights to {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)
print("Weight merge complete! Model is ready for GGUF quantization.")